# 01 — Data Ingestion

Explore and validate all three data sources before running the full ETL pipeline:

1. **CCEE PLD** via IPEADATA (weekly spot prices, all 4 subsystems)
2. **ONS grid data** via dados.ons.org.br CKAN API (reservoir, ENA, load, generation, interconnection)
3. **Open-Meteo weather** (daily weather for 4 representative cities)

**Primary goal of this notebook:** Inspect the raw API responses, confirm column names and data types, then update the placeholder column mappings in `etl/bronze.py`.

In [1]:
import sys
sys.path.insert(0, '..')

import time
import requests
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Setup complete')

Setup complete


## 1. ONS CMO Weekly — PLD proxy

The IPEADATA PLD series (`ELETROENERGIA_PLDSECO` etc.) no longer return data.
We now source weekly CMO (Custo Marginal de Operação ≈ PLD) directly from
**ONS Open Data**: `dados.ons.org.br/dataset/cmo-semanal`

CSV files are hosted on S3 per year. Column `val_cmomediasemanal` = weekly average CMO (R$/MWh).
Available from 2005 onwards.

### 1.1 Fetch a sample (2024)

In [2]:
ONS_CMO_TEMPLATE = (
    "https://ons-aws-prod-opendata.s3.amazonaws.com/dataset/cmo_se/CMO_SEMANAL_{year}.csv"
)
SUBSYSTEM_MAP = {"SE": "SE/CO", "S": "S", "NE": "NE", "N": "N"}

# Fetch 2024 as a sample
url_2024 = ONS_CMO_TEMPLATE.format(year=2024)
df_sample = pd.read_csv(url_2024, sep=";")
print(f"Shape: {df_sample.shape}")
print(f"Columns: {df_sample.columns.tolist()}")
print(f"Subsystems: {df_sample['id_subsistema'].unique().tolist()}")
print()
df_sample.head(8)

Shape: (208, 7)
Columns: ['id_subsistema', 'nom_subsistema', 'din_instante', 'val_cmomediasemanal', 'val_cmoleve', 'val_cmomedia', 'val_cmopesada']
Subsystems: ['N', 'NE', 'S', 'SE']



,id_subsistema,nom_subsistema,din_instante,val_cmomediasemanal,val_cmoleve,val_cmomedia,val_cmopesada
0,N,NORTE,2024-01-05,0.00,0.00,0.00,0.00
1,NE,NORDESTE,2024-01-05,0.00,0.00,0.00,0.00
2,S,SUL,2024-01-05,0.00,0.00,0.00,0.00
3,SE,SUDESTE,2024-01-05,0.00,0.00,0.00,0.00
4,N,NORTE,2024-01-12,0.00,0.00,0.00,0.00
5,NE,NORDESTE,2024-01-12,0.00,0.00,0.00,0.00
6,S,SUL,2024-01-12,0.00,0.00,0.00,0.00
7,SE,SUDESTE,2024-01-12,0.00,0.00,0.00,0.00


In [3]:
def fetch_cmo_year(year: int) -> pd.DataFrame:
    """Download ONS CMO CSV for a given year and normalize to project schema."""
    url = ONS_CMO_TEMPLATE.format(year=year)
    resp = requests.get(url, timeout=60)
    if resp.status_code == 404:
        print(f"  {year}: not found (404)")
        return pd.DataFrame()
    resp.raise_for_status()
    df = pd.read_csv(url, sep=";")
    df = df.rename(columns={
        "din_instante": "week_start",
        "val_cmomediasemanal": "pld_brl_mwh",
        "id_subsistema": "subsystem_code",
    })
    df["week_start"] = pd.to_datetime(df["week_start"])
    df["pld_brl_mwh"] = pd.to_numeric(df["pld_brl_mwh"], errors="coerce")
    df["subsystem"] = df["subsystem_code"].map(SUBSYSTEM_MAP)
    return df[["week_start", "pld_brl_mwh", "subsystem"]].dropna()

# Quick check: what's the date range in 2024?
df_2024 = fetch_cmo_year(2024)
print(f"2024 — {len(df_2024)} rows, {df_2024.week_start.min().date()} → {df_2024.week_start.max().date()}")
print(df_2024.groupby("subsystem")["pld_brl_mwh"].describe().round(2))

2024 — 208 rows, 2024-01-05 → 2024-12-27
           count  mean    std  min  25%   50%   75%    max
subsystem                                                 
N          52.00 99.45 164.84 0.00 0.02 12.90 95.68 624.81
NE         52.00 99.45 164.84 0.00 0.02 12.90 95.68 624.81
S          52.00 99.86 164.60 0.00 0.06 16.83 95.68 624.81
SE/CO      52.00 99.86 164.60 0.00 0.06 16.83 95.68 624.81


### 1.2 Fetch all available years (2005–2025) and compare subsystems

In [4]:
frames = []
for year in range(2005, 2026):
    df = fetch_cmo_year(year)
    if not df.empty:
        frames.append(df)
    time.sleep(0.1)

pld_all = pd.concat(frames, ignore_index=True)
print(f"Total rows: {len(pld_all)}")
print(f"Date range: {pld_all.week_start.min().date()} → {pld_all.week_start.max().date()}")
pld_all.groupby("subsystem")["pld_brl_mwh"].describe().round(2)

Total rows: 4380
Date range: 2005-01-07 → 2025-12-26


,count,mean,std,min,25%,50%,75%,max
subsystem,,,,,,,,
N,1095.00,167.82,269.69,0.00,3.47,85.72,214.60,3044.45
NE,1095.00,187.63,275.16,0.00,16.48,101.41,250.60,3044.45
S,1095.00,203.40,312.79,0.00,28.60,101.41,241.76,3044.45
SE/CO,1095.00,205.54,312.96,0.00,30.18,103.09,247.90,3044.45


In [5]:
# Visualize PLD history SE/CO
pld_seco = pld_all[pld_all["subsystem"] == "SE/CO"].sort_values("week_start")
print(f"SE/CO — {len(pld_seco)} rows")
print(f"Date range: {pld_seco.week_start.min().date()} → {pld_seco.week_start.max().date()}")
print(f"Null values: {pld_seco.isna().sum().to_dict()}")
pld_seco.tail(10)

SE/CO — 1095 rows
Date range: 2005-01-07 → 2025-12-26
Null values: {'week_start': 0, 'pld_brl_mwh': 0, 'subsystem': 0}


,week_start,pld_brl_mwh,subsystem
4343,2025-10-24,292.61,SE/CO
4347,2025-10-31,302.79,SE/CO
4351,2025-11-07,349.82,SE/CO
4355,2025-11-14,305.15,SE/CO
4359,2025-11-21,326.60,SE/CO
4363,2025-11-28,324.17,SE/CO
4367,2025-12-05,337.53,SE/CO
4371,2025-12-12,305.91,SE/CO
4375,2025-12-19,300.87,SE/CO
4379,2025-12-26,252.32,SE/CO


In [6]:
# Visualize PLD/CMO history SE/CO
fig = px.line(pld_seco, x='week_start', y='pld_brl_mwh',
              title='CMO SE/CO — Weekly (R$/MWh) | source: ONS cmo-semanal',
              template='plotly_dark')
fig.update_layout(xaxis_title='Week', yaxis_title='R$/MWh')
fig.show()

### 1.3 Fetch all 4 subsystems and compare

Update the series codes dict below based on the discovery step.

In [7]:
# pld_all already contains all 4 subsystems from the fetch above
# Plot all subsystems for comparison
fig = px.line(pld_all, x='week_start', y='pld_brl_mwh', color='subsystem',
              title='CMO — All 4 Subsystems (R$/MWh) | source: ONS cmo-semanal',
              template='plotly_dark',
              color_discrete_map={'SE/CO': '#636EFA', 'S': '#EF553B', 'NE': '#00CC96', 'N': '#AB63FA'})
fig.update_layout(xaxis_title='Week', yaxis_title='R$/MWh', hovermode='x unified')
fig.show()

In [8]:
# Verify the IPEADATA_PLD_SERIES dict in etl/collect.py matches the codes above
# then plot all 4 subsystems
fig = px.line(pld_all, x='week_start', y='pld_brl_mwh', color='subsystem',
              title='PLD — All 4 Subsystems (R$/MWh)', template='plotly_dark',
              color_discrete_map={'SE/CO': '#636EFA', 'S': '#EF553B', 'NE': '#00CC96', 'N': '#AB63FA'})
fig.update_layout(xaxis_title='Week', yaxis_title='R$/MWh', hovermode='x unified')
fig.show()

---
## 2. ONS Open Data — dados.ons.org.br

The ONS CKAN API: `GET https://dados.ons.org.br/api/3/action/datastore_search?resource_id=<id>`

No authentication required. Paginate with `limit` + `offset`.

### 2.1 Discover available datasets

In [9]:
ONS_BASE = 'https://dados.ons.org.br/api/3/action'

# List all packages (datasets) on the portal
resp = requests.get(f'{ONS_BASE}/package_list', timeout=30)
resp.raise_for_status()
packages = resp.json()['result']
print(f'Total packages on dados.ons.org.br: {len(packages)}')

# Filter for likely relevant datasets
keywords = ['reservat', 'ena', 'carga', 'geracao', 'intercambio', 'pld', 'submercado']
relevant = [p for p in packages if any(k in p.lower() for k in keywords)]
print(f'Relevant packages: {len(relevant)}')
for p in sorted(relevant):
    print(' ', p)

Total packages on dados.ons.org.br: 80
Relevant packages: 25
  capacidade-geracao
  carga-energia
  carga-energia-programada
  carga-energia-verificada
  carga-mensal
  cargaglobal-roraima
  curva-carga
  ear-diario-por-ree-reservatorio-equivalente-de-energia
  ear-diario-por-reservatorio
  ena-diario-por-bacia
  ena-diario-por-ree-reservatorio-equivalente-de-energia
  ena-diario-por-reservatorio
  ena-diario-por-subsistema
  geracao-exportacao-internacional
  geracao-termica-despacho-2
  geracao-usina-2
  geracao_itaipu
  ind-disponibilidade-geracao-sin
  ind_disponibilidade_fgeracao_uge_anual
  ind_disponibilidade_fgeracao_uge_mensal
  intercambio-internacional
  intercambio-nacional
  intercambio_modalidade
  interrupcao_carga
  reservatorio


In [10]:
def ons_package_info(package_name: str) -> pd.DataFrame:
    """Get resource IDs and descriptions for an ONS package."""
    resp = requests.get(f'{ONS_BASE}/package_show', params={'id': package_name}, timeout=30)
    if not resp.ok:
        print(f'ERROR: {resp.status_code} for {package_name}')
        return pd.DataFrame()
    resources = resp.json()['result'].get('resources', [])
    return pd.DataFrame([{
        'id': r['id'],
        'name': r.get('name', ''),
        'description': r.get('description', '')[:80],
        'format': r.get('format', ''),
    } for r in resources])

# Explore the most promising packages
for pkg in relevant[:8]:
    info = ons_package_info(pkg)
    if not info.empty:
        print(f'\n=== {pkg} ===')
        print(info.to_string(index=False))


=== capacidade-geracao ===
                                  id                     name                                                                      description  format
54755e75-fcd9-4e90-8821-d1dc3d41bcd1      Dicionário de Dados --------------------------------------------------------------------------------     PDF
a6412542-f2ce-408e-b51d-19a48cc50b62       Capacidade_Geracao                                                                                      CSV
515cc325-6976-4c32-b3bf-48d035b70277       Capacidade_Geracao                                                                                     XLSX
03aff81f-4a46-40f3-b076-463458c94356       Capacidade_Geracao                                                                 Extensão PARQUET PARQUET
b1f422cb-af4d-4b06-9fba-886db50eef8c Dicionário de Dados Json                                              Dicionário de Dados no formato JSON    JSON

=== carga-energia ===
                                  id       

### 2.2 Fetch a sample from each key resource

Update the resource IDs below from the discovery step above, then check column names.

In [11]:
def ons_sample(resource_id: str, n: int = 5) -> pd.DataFrame:
    """Fetch a small sample from an ONS resource."""
    resp = requests.get(
        f'{ONS_BASE}/datastore_search',
        params={'resource_id': resource_id, 'limit': n},
        timeout=30
    )
    if not resp.ok:
        print(f'ERROR {resp.status_code}: {resp.text[:200]}')
        return pd.DataFrame()
    result = resp.json().get('result', {})
    df = pd.DataFrame(result.get('records', []))
    print(f'Total records in resource: {result.get("total", "?")} | Columns: {list(df.columns)}')
    return df

In [12]:
# ✏️  UPDATE: paste the actual resource_id values found above for each dataset
# The resource IDs are UUIDs like 'xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx'

# --- RESERVOIR / ENA ---
print('=== RESERVOIR / ENA ===')
# Try the package name discovered above, replace with actual resource_id UUID
# resource_id_reservoir = 'PASTE-UUID-HERE'
# df_res_sample = ons_sample(resource_id_reservoir)
# df_res_sample

=== RESERVOIR / ENA ===


In [13]:
# Once you have real resource IDs, run this block for each resource.
# The goal is to identify:
#   - Date column name (e.g., dat_referencia, dat_semana_inicio, DatReferencia)
#   - Subsystem column name (e.g., nom_submercado, NomSubmercado, IdSubmercado)
#   - Value column name (e.g., val_pct_volume_util, ValPctVolumeUtil)
#
# After identifying all column names, update etl/bronze.py accordingly.

# Example workflow (uncomment and run after finding resource IDs):
# 
# resource_id_ena = 'PASTE-UUID-HERE'
# df_ena = ons_sample(resource_id_ena)
# print('ENA columns:', df_ena.columns.tolist())
# df_ena.head()

### 2.3 Reservoir data — full fetch and exploration

In [14]:
def ons_fetch_all(resource_id: str) -> pd.DataFrame:
    """Paginate through the ONS CKAN API and return ALL records."""
    offset, limit, records = 0, 5000, []
    while True:
        resp = requests.get(
            f'{ONS_BASE}/datastore_search',
            params={'resource_id': resource_id, 'limit': limit, 'offset': offset},
            timeout=60
        )
        resp.raise_for_status()
        batch = resp.json()['result']['records']
        if not batch:
            break
        records.extend(batch)
        print(f'  fetched {len(records)} records so far...', end='\r')
        offset += limit
        if len(batch) < limit:
            break
        time.sleep(0.2)
    print(f'  Done: {len(records)} total records      ')
    return pd.DataFrame(records)

# Uncomment once you have the reservoir resource_id:
# df_reservoir = ons_fetch_all(resource_id_reservoir)
# print(df_reservoir.dtypes)
# df_reservoir.head()

In [15]:
# After fetching, inspect the actual column names and update etl/bronze.py
# ============================================================
# COLUMN MAPPING TEMPLATE (fill in after running the fetches above):
# ============================================================

# RESERVOIR (_build_reservoir in etl/bronze.py):
#   date col   → ???  (e.g., 'dat_semana_inicio' or 'DatReferencia')
#   subsystem  → ???  (e.g., 'nom_submercado' or 'NomSubmercado')
#   reservoir% → ???  (e.g., 'val_pct_volume_util' or 'VlrPctVolumeUtil')

# ENA (_build_reservoir, ENA part):
#   date col   → ???
#   subsystem  → ???
#   ena_gwh    → ???  (e.g., 'val_ena_bruta_gwh')

# LOAD (_build_load):
#   date col   → ???
#   subsystem  → ???
#   load_mwh   → ???  (e.g., 'val_carga_energia')

# GENERATION (_build_generation):
#   date col   → ???
#   source     → ???  (e.g., 'nom_tipo_geracao') — expected values: HIDRO, EOLICA, SOLAR, TERMELETRICA, NUCLEAR
#   gen_mw     → ???  (e.g., 'val_geracao_mwmed')

# INTERCONNECTION (_build_interconnection):
#   date col      → ???
#   from_subsys   → ???
#   to_subsys     → ???
#   flow_mwh      → ???

print('Fill in the mapping above, then update etl/bronze.py')

Fill in the mapping above, then update etl/bronze.py


### 2.4 ENA data exploration (if available)

In [16]:
# After fetching ENA data, explore the ENA anomaly concept:
# ENA anomaly = ENA this week vs. historical average for same week-of-year
# Values < 1.0 = drought conditions → PLD will likely spike

# Uncomment once df_ena is available:
# 
# # Rename to standard names first
# df_ena_clean = df_ena.rename(columns={
#     'dat_semana_inicio': 'week_start',  # ← actual column name
#     'nom_submercado': 'subsystem',       # ← actual column name
#     'val_ena_bruta_gwh': 'ena_gwh',      # ← actual column name
# })
# df_ena_clean['week_start'] = pd.to_datetime(df_ena_clean['week_start'])
# df_ena_clean['week_of_year'] = df_ena_clean['week_start'].dt.isocalendar().week
# 
# # Compute historical average ENA for each subsystem × week_of_year
# hist_avg = df_ena_clean.groupby(['subsystem', 'week_of_year'])['ena_gwh'].mean().reset_index()
# hist_avg.columns = ['subsystem', 'week_of_year', 'ena_hist_avg']
# 
# df_ena_merged = df_ena_clean.merge(hist_avg, on=['subsystem', 'week_of_year'])
# df_ena_merged['ena_anomaly'] = df_ena_merged['ena_gwh'] / df_ena_merged['ena_hist_avg']
# 
# # Plot ENA anomaly for SE/CO
# seco = df_ena_merged[df_ena_merged['subsystem'] == 'SE/CO'].sort_values('week_start')
# fig = px.line(seco, x='week_start', y='ena_anomaly',
#               title='ENA Anomaly — SE/CO (1.0 = historical average | <1.0 = drought)',
#               template='plotly_dark')
# fig.add_hline(y=1.0, line_dash='dash', line_color='gray')
# fig.add_hline(y=0.7, line_dash='dash', line_color='red', annotation_text='Drought threshold')
# fig.show()

print('Uncomment after fetching ENA data')

Uncomment after fetching ENA data


---
## 3. Open-Meteo Weather

Daily weather data for 4 representative cities (one per subsystem). Free, no API key.

In [17]:
import openmeteo_requests
import requests_cache
from retry_requests import retry

cache_session = requests_cache.CachedSession('.weather_cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
client = openmeteo_requests.Client(session=retry_session)

CITIES = {
    'SE/CO': {'lat': -23.55, 'lon': -46.63, 'name': 'São Paulo'},
    'S':     {'lat': -25.43, 'lon': -49.27, 'name': 'Curitiba'},
    'NE':    {'lat': -3.72,  'lon': -38.54, 'name': 'Fortaleza'},
    'N':     {'lat': -1.45,  'lon': -48.50, 'name': 'Belém'},
}

print('Testing Open-Meteo for São Paulo (SE/CO), last 30 days...')
city = CITIES['SE/CO']
responses = client.weather_api(
    'https://archive-api.open-meteo.com/v1/archive',
    params={
        'latitude': city['lat'], 'longitude': city['lon'],
        'start_date': '2024-01-01', 'end_date': '2024-01-31',
        'daily': ['precipitation_sum', 'temperature_2m_mean', 'wind_speed_10m_mean'],
        'timezone': 'America/Sao_Paulo',
    }
)
daily = responses[0].Daily()
dates = pd.date_range(
    start=pd.to_datetime(daily.Time(), unit='s'),
    end=pd.to_datetime(daily.TimeEnd(), unit='s'),
    freq=pd.Timedelta(seconds=daily.Interval()),
    inclusive='left'
)
df_weather = pd.DataFrame({
    'date': dates,
    'precip_mm': daily.Variables(0).ValuesAsNumpy(),
    'temp_c': daily.Variables(1).ValuesAsNumpy(),
    'wind_speed_kmh': daily.Variables(2).ValuesAsNumpy(),
})
print(f'Shape: {df_weather.shape} | Date range: {df_weather.date.min().date()} → {df_weather.date.max().date()}')
df_weather.head()

Testing Open-Meteo for São Paulo (SE/CO), last 30 days...
Shape: (31, 4) | Date range: 2024-01-01 → 2024-01-31


,date,precip_mm,temp_c,wind_speed_kmh
0,2024-01-01 03:00:00,0.30,21.27,14.73
1,2024-01-02 03:00:00,2.50,21.96,11.77
2,2024-01-03 03:00:00,32.70,22.00,9.63
3,2024-01-04 03:00:00,3.40,21.48,16.03
4,2024-01-05 03:00:00,1.50,21.02,12.46


In [18]:
# Weekly aggregation (what bronze.py does via DuckDB)
df_weather['week_start'] = df_weather['date'].dt.to_period('W').dt.start_time
weekly = df_weather.groupby('week_start').agg(
    precip_mm_sum=('precip_mm', 'sum'),
    temp_c_avg=('temp_c', 'mean'),
    wind_speed_avg=('wind_speed_kmh', 'mean'),
).reset_index()
print('Weekly weather (São Paulo):')
weekly

Weekly weather (São Paulo):


,week_start,precip_mm_sum,temp_c_avg,wind_speed_avg
0,2024-01-01,41.10,21.70,12.11
1,2024-01-08,67.80,23.51,11.01
2,2024-01-15,36.70,24.43,13.57
3,2024-01-22,43.70,19.01,14.13
4,2024-01-29,7.40,22.44,8.16


In [19]:
# Verify all 4 cities are accessible
for subsystem, city in CITIES.items():
    resp = client.weather_api(
        'https://archive-api.open-meteo.com/v1/archive',
        params={
            'latitude': city['lat'], 'longitude': city['lon'],
            'start_date': '2024-01-01', 'end_date': '2024-01-07',
            'daily': ['precipitation_sum'],
            'timezone': 'America/Sao_Paulo',
        }
    )
    n = resp[0].Daily().Variables(0).ValuesAsNumpy().shape[0]
    print(f'{subsystem} ({city["name"]}): {n} days ✓')

SE/CO (São Paulo): 7 days ✓
S (Curitiba): 7 days ✓
NE (Fortaleza): 7 days ✓
N (Belém): 7 days ✓


---
## 4. Quick sanity check — run collect.py for a small date range

In [20]:
# Once you've verified the API responses and updated the series codes + column names:
# !python -m etl.collect --start 2022-01-01 --end 2024-12-31

# To collect only PLD first (fastest to verify):
# !python -m etl.collect --start 2022-01-01 --end 2024-12-31 --only pld

# Then inspect what was saved:
import os
from pathlib import Path

raw_dir = Path('../data/raw')
if raw_dir.exists():
    for f in sorted(raw_dir.rglob('*.parquet')):
        size_kb = f.stat().st_size / 1024
        print(f'{f.relative_to(raw_dir)!s:60s}  {size_kb:.1f} KB')
else:
    print('No raw data yet — run etl.collect first')

In [21]:
# After collecting, verify the PLD raw parquet
import duckdb

pld_raw = '../data/raw/pld/**/*.parquet'
try:
    con = duckdb.connect()
    df = con.execute(f"SELECT * FROM read_parquet('{pld_raw}', hive_partitioning=true) ORDER BY week_start DESC LIMIT 20").fetchdf()
    con.close()
    print(f'PLD raw: {len(df)} rows loaded')
    df
except Exception as e:
    print(f'Not available yet: {e}')

Not available yet: IO Error: No files found that match the pattern "../data/raw/pld/**/*.parquet"

LINE 1: SELECT * FROM read_parquet('../data/raw/pld/**/*.parquet', hive_partitioning...
                      ^


---
## 5. Summary: checklist before running the full pipeline

Complete all items below before running `python -m etl.run_pipeline`:

- [x] **PLD/CMO source changed** — now fetches from ONS `cmo-semanal` (S3 CSVs per year, 2005+); IPEADATA series were deprecated. `etl/collect.py` updated.
- [ ] Found ONS resource IDs for: reservoir, ENA, load, generation, interconnection → updated `ONS_RESOURCES` in `etl/collect.py`
- [ ] Identified actual column names from ONS API responses → updated column mappings in `etl/bronze.py`
- [x] Verified Open-Meteo works for all 4 cities ✓
- [ ] Ran a small collect test (`--start 2022-01-01 --end 2024-12-31`) and confirmed data looks correct

Once all boxes are checked:
```bash
cd ..
python -m etl.run_pipeline --start 2005-01-01 --end 2024-12-31
```